In [ ]:
"""
CombiningMultipleTrialsWithinSession.py
A script for combining multiple trials within a session for the Spatial Modulation Index (SMI) analysis (same FOV)
Input: 2p and VR data processed using preprocess.py for the same session (but each recording was only for 60 trials)
Output: combined 2p and VR data for the whole session (50-200 trials)

JSY, 10/03/25
"""
import sys
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation")

import os
import glob
import numpy as np
import datetime
from helper import files, read_xml

c:\Users\jasmineyeo\AppData\Local\anaconda3\envs\JSY_SpMod\Lib\site-packages\oasis\functions.py:13: UserWarning: Could not find cvxpy. Don't worry, you can still use OASIS, just not the slower interior point methods we compared to in the papers.
  warn("Could not find cvxpy. Don't worry, you can still use OASIS, " +


In [ ]:
def deconcat_suite2p_output(session_directory):
    """
    Deconcatenate the suite2p output files for each trial within the session
    Input: session_directory - folder containing a single suite2p output folder for all trials within the session
    Output: deconcatenated F, Fneu, spks, iscell, stat, ops for each trial
    """
    
    # Find all trial directories
    trial_dir = sorted(glob.glob(os.path.join(session_directory, "TSeries*")))
    num_trials = len(trial_dir)
    num_frames = np.zeros(num_trials, dtype=int)
    
    print(f"Number of trials detected: {num_trials}")
    
    for i in range(num_trials):
        trial_directory = trial_dir[i]
        xml_directory = glob.glob(os.path.join(trial_directory, "*.xml"))[0]
        xml_dict = read_xml(xml_directory)
        num_frames[i] = int(xml_dict['num_frames'])
        print(f"Number of frames in trial {i+1}: {num_frames[i]}")
    
    # Calculate cumulative end frames
    cumulative_frames = np.cumsum(num_frames)
    print(f"Cumulative frames from XML: {cumulative_frames}")

    # Load twoP data    
    F = np.load(os.path.join(session_directory, r'suite2p/plane0/F.npy'), allow_pickle=True)
    Fneu = np.load(os.path.join(session_directory, r'suite2p/plane0/Fneu.npy'), allow_pickle=True)
    iscell = np.load(os.path.join(session_directory, r'suite2p/plane0/iscell.npy'), allow_pickle=True)
    stat = np.load(os.path.join(session_directory, r'suite2p/plane0/stat.npy'), allow_pickle=True)
    ops = np.load(os.path.join(session_directory, r'suite2p/plane0/ops.npy'), allow_pickle=True).item()
    spks = np.load(os.path.join(session_directory, r'suite2p/plane0/spks.npy'), allow_pickle=True)

    usecells = iscell[:, 0] == 1

    twoP_data = {}
    twoP_data['F'] = F[usecells, :]
    twoP_data['Fneu'] = Fneu[usecells, :]
    twoP_data['stat'] = stat[usecells]
    twoP_data['ops'] = ops
    twoP_data['spks'] = spks[usecells, :]
    twoP_data['iscell'] = iscell[usecells, :]
    
    #  Verify frame counts BEFORE splitting
    total_frames_in_suite2p = twoP_data['F'].shape[1]
    total_frames_expected = int(cumulative_frames[-1])
    
    print(f"\nVerifying frame counts:")
    print(f"  Suite2p actual: {total_frames_in_suite2p} frames")
    print(f"  XML expected: {total_frames_expected} frames")
    
    if total_frames_in_suite2p != total_frames_expected:
        print(f"  ⚠️ Mismatch of {total_frames_in_suite2p - total_frames_expected} frames detected!")
        print(f"  Adjusting split points to match actual Suite2p data...")
        
        # Proportionally adjust num_frames to match actual data
        ratio = total_frames_in_suite2p / total_frames_expected
        adjusted_num_frames = (num_frames * ratio).astype(int)
        
        # Ensure the last trial gets any remaining frames
        adjusted_num_frames[-1] = total_frames_in_suite2p - np.sum(adjusted_num_frames[:-1])
        
        cumulative_frames = np.cumsum(adjusted_num_frames)
        print(f"  Adjusted frame counts per trial: {adjusted_num_frames}")
        print(f"  Adjusted cumulative frames: {cumulative_frames}")
    else:
        print(f"  Frame counts match perfectly!")
    
    # Split and save data for each trial
    for i in range(num_trials):
        print(f"\nProcessing trial {i+1}/{num_trials}...")
        
        # Create output directory
        output_dir = os.path.join(trial_dir[i], 'suite2p/plane0/')
        os.makedirs(output_dir, exist_ok=True)
        
        # Determine frame range for this trial
        start_frame = 0 if i == 0 else int(cumulative_frames[i-1])
        end_frame = int(cumulative_frames[i])
        print(f"  Saving frames {start_frame} to {end_frame}")
        
        # Process each data type
        for key in twoP_data.keys():
            if key in ['F', 'Fneu', 'spks']:
                # Split temporal data by frames
                data_split = twoP_data[key][:, start_frame:end_frame]
                print(f"  {key} shape: {data_split.shape}")
            else:
                # Non-temporal data (stat, ops, iscell) - same for all trials
                data_split = twoP_data[key]
            
            # Save the split data
            np.save(os.path.join(output_dir, f'{key}.npy'), data_split)
        
        print(f"  Saved to {output_dir}")
    
    print("\nDeconcatenation complete!")
    print(f"Total frames distributed: {int(cumulative_frames[-1])}")

In [3]:
concat_suite2p_direcotry = r"F:\2P\spmod\JSY052_ChrnoicImaging\251012_JSY_JSY052_SpatialModulation_Day4"
deconcat_suite2p_output(concat_suite2p_direcotry)

Number of trials detected: 2
Number of frames in trial 1: 24001
Number of frames in trial 2: 15528
Cumulative frames from XML: [24001 39529]

Verifying frame counts:
  Suite2p actual: 39527 frames
  XML expected: 39529 frames
  ⚠️ Mismatch of -2 frames detected!
  Adjusting split points to match actual Suite2p data...
  Adjusted frame counts per trial: [23999 15528]
  Adjusted cumulative frames: [23999 39527]

Processing trial 1/2...
  Saving frames 0 to 23999
  F shape: (1224, 23999)
  Fneu shape: (1224, 23999)
  spks shape: (1224, 23999)
  Saved to F:\2P\spmod\JSY052_ChrnoicImaging\251012_JSY_JSY052_SpatialModulation_Day4\TSeries-10122025-1212-001\suite2p/plane0/

Processing trial 2/2...
  Saving frames 23999 to 39527
  F shape: (1224, 15528)
  Fneu shape: (1224, 15528)
  spks shape: (1224, 15528)
  Saved to F:\2P\spmod\JSY052_ChrnoicImaging\251012_JSY_JSY052_SpatialModulation_Day4\TSeries-10122025-1212-002\suite2p/plane0/

Deconcatenation complete!
Total frames distributed: 39527


In [ ]:
def combine_session_trials(session_directory, save_output=True):
    """
    After preprocessing each trial individually with preprocess.py, this script:
    1. Loads all preprocessed trials from a session
    2. Verifies cell matching across trials (should be identical after concatenated Suite2p)
    3. Identifies cells that are reliable across ALL trials
    4. Combines spatial activity data across trials
    5. Saves combined session data for SMI analysis 
       
    Parameters:
    -----------
    session_directory : str
        Path to session directory containing TSeries folders with preprocessed data
    save_output : bool
        Whether to save the combined data to disk
        
    Returns:
    --------
    combined_data : dict
        Dictionary containing combined session data with session-level reliability
    """
    
    # Find all trial directories
    trial_dirs = sorted(glob.glob(os.path.join(session_directory, "TSeries*")))
    num_trials = len(trial_dirs)
    
    if num_trials == 0:
        raise ValueError(f"No TSeries directories found in {session_directory}")
    
    print(f"\nFound {num_trials} trial recordings")
    
    # Load all preprocessed data
    preproc_data = []
    for i, trial_dir in enumerate(trial_dirs):
        preproc_files = glob.glob(os.path.join(trial_dir, "*preproc.h5"))
        
        if not preproc_files:
            raise ValueError(f"No preproc.h5 file found in {trial_dir}")
        
        data = files.read_h5(preproc_files[0])
        preproc_data.append(data)
        
        n_cells = data['spatial_activity'].shape[0]
        n_bins = data['spatial_activity'].shape[1]
        n_trials_in_recording = data['spatial_activity'].shape[2]
        
        print(f"\nRecording {i+1}: {os.path.basename(trial_dir)}")
        print(f"  Loaded: {os.path.basename(preproc_files[0])}")
        print(f"  Cells: {n_cells}, Bins: {n_bins}, Trials: {n_trials_in_recording}")
    
    # Verify all recordings have the same number of cells
    num_cells_per_recording = [data['spatial_activity'].shape[0] for data in preproc_data]
    
    if len(set(num_cells_per_recording)) != 1:
        raise ValueError(
            f"Different number of cells across recordings: {num_cells_per_recording}\n"
            "This suggests Suite2p was not run on concatenated data.\n"
            "Please re-run Suite2p on concatenated recordings."
        )
    
    n_cells = num_cells_per_recording[0]
    print(f"\n✓ All recordings have {n_cells} cells (Suite2p concatenation successful)")
    
    # Verify bin centers are identical
    bin_centers_match = all(
        np.allclose(preproc_data[0]['bin_centers'], data['bin_centers']) 
        for data in preproc_data
    )
    
    if not bin_centers_match:
        print("WARNING: Bin centers differ across recordings!")
    
    print("\n" + "="*80)
    print("STEP 1: CONCATENATING SPATIAL ACTIVITY DATA")
    print("="*80)
    
    # Get all spatial activity arrays
    spatial_activity_list = [data['spatial_activity'] for data in preproc_data]
    norm_spatial_activity_list = [data['norm_spatial_activity'] for data in preproc_data]

    # Check shapes
    print("\nSpatial activity shapes:")
    for i, arr in enumerate(spatial_activity_list):
        print(f"  Recording {i+1}: {arr.shape} (cells × bins × trials)")

    # Verify trials dimension is consistent
    trials_per_recording = [arr.shape[2] for arr in spatial_activity_list]
    if len(set(trials_per_recording)) != 1:
        raise ValueError(f"Different number of trials across recordings: {trials_per_recording}")

    # Concatenate along bins dimension (axis=1)
    combined_spatial_activity = np.concatenate(spatial_activity_list, axis=1)
    combined_norm_spatial_activity = np.concatenate(norm_spatial_activity_list, axis=1)

    total_bins = combined_spatial_activity.shape[1]
    bins_per_recording = [arr.shape[1] for arr in spatial_activity_list]

    print(f"\nCombined spatial activity shape: {combined_spatial_activity.shape}")
    print(f"  ({combined_spatial_activity.shape[0]} cells × {total_bins} bins × {combined_spatial_activity.shape[2]} trials)")
    print(f"  Bins per recording: {bins_per_recording}")

    # Concatenate along trials dimension (axis=1)
    combined_spatial_activity = np.concatenate(spatial_activity_list, axis=1)
    combined_norm_spatial_activity = np.concatenate(norm_spatial_activity_list, axis=1)

    total_trials = combined_spatial_activity.shape[1]
    trials_per_recording = [arr.shape[1] for arr in spatial_activity_list]

    print(f"\nCombined spatial activity shape: {combined_spatial_activity.shape}")
    print(f"  ({combined_spatial_activity.shape[0]} cells × {total_trials} trials × {combined_spatial_activity.shape[2]} bins)")
    print(f"  Trials per recording: {trials_per_recording}")

    total_trials = combined_spatial_activity.shape[2]
    trials_per_recording = [data['spatial_activity'].shape[2] for data in preproc_data]
    
    print(f"\nCombined spatial activity shape: {combined_spatial_activity.shape}")
    print(f"  ({n_cells} cells × {combined_spatial_activity.shape[1]} bins × {total_trials} total trials)")
    print(f"  Trials per recording: {trials_per_recording}")
    
    print("\n" + "="*80)
    print("STEP 2: AVERAGING PER-TRIAL STATISTICS")
    print("="*80)
    
    # Average statistics across recordings
    avg_cc_list = [data['avg_cc'] for data in preproc_data]
    cohen_d_list = [data['cohen_d'] for data in preproc_data]
    
    avg_cc_session = np.mean(avg_cc_list, axis=0)
    cohen_d_session = np.mean(cohen_d_list, axis=0)
    
    print(f"\nAveraged correlation coefficients across {num_trials} recordings")
    print(f"  Mean CC range: [{np.min(avg_cc_session):.3f}, {np.max(avg_cc_session):.3f}]")
    print(f"\nAveraged Cohen's d across {num_trials} recordings")
    print(f"  Mean Cohen's d range: [{np.min(cohen_d_session):.3f}, {np.max(cohen_d_session):.3f}]")
    
    print("\n" + "="*80)
    print("STEP 3: FINDING CELLS RELIABLE ACROSS ALL RECORDINGS")
    print("="*80)
    
    # Store per-recording reliability for reference
    trial_reliable_cells_list = [data['reliable_cells'] for data in preproc_data]
    trial_combined_reliable_list = [data['combined_reliable'] for data in preproc_data]
    
    # Find cells reliable in ALL recordings (strict criterion)
    session_reliable_cells = np.logical_and.reduce(trial_reliable_cells_list)
    session_combined_reliable = np.logical_and.reduce(trial_combined_reliable_list)
    
    session_reliable_cell_indices = np.where(session_reliable_cells)[0]
    session_combined_reliable_indices = np.where(session_combined_reliable)[0]
    
    print(f"\nPer-recording reliability:")
    for i, reliable in enumerate(trial_reliable_cells_list):
        print(f"  Recording {i+1}: {np.sum(reliable)} reliable cells")
    
    print(f"\nCells reliable in ALL recordings: {np.sum(session_reliable_cells)}")
    
    print(f"\nPer-recording combined reliability:")
    for i, reliable in enumerate(trial_combined_reliable_list):
        print(f"  Recording {i+1}: {np.sum(reliable)} combined reliable cells")
    
    print(f"\nCells combined-reliable in ALL recordings: {np.sum(session_combined_reliable)}")
    
    print("\n" + "="*80)
    print("STEP 4: ASSEMBLING COMBINED DATA STRUCTURE")
    print("="*80)
    
    # Create trial info metadata
    trial_info = []
    for i, (trial_dir, n_trials) in enumerate(zip(trial_dirs, trials_per_recording)):
        trial_info.append({
            'recording_num': i + 1,
            'trial_directory': trial_dir,
            'n_trials': n_trials,
            'vr_filepath': preproc_data[i].get('vr_filepath', 'unknown'),
            'twop_filepath': preproc_data[i].get('twop_filepath', 'unknown')
        })
    
    # Assemble combined data dictionary
    combined_data = {
        # ===== CONCATENATED DATA (trials combined) =====
        'spatial_activity': combined_spatial_activity,
        'norm_spatial_activity': combined_norm_spatial_activity,
        
        # ===== SESSION-LEVEL RELIABILITY (cells reliable in ALL recordings) =====
        'session_reliable_cells': session_reliable_cells,
        'session_combined_reliable': session_combined_reliable,
        'session_reliable_cell_indices': session_reliable_cell_indices,
        'session_combined_reliable_indices': session_combined_reliable_indices,
        
        # ===== AVERAGED STATISTICS (from individual recordings) =====
        'avg_cc_session': avg_cc_session,
        'cohen_d_session': cohen_d_session,
        
        # ===== PER-RECORDING RELIABILITY (for reference) =====
        'trial_reliable_cells_list': trial_reliable_cells_list,
        'trial_combined_reliable_list': trial_combined_reliable_list,
        
        # ===== KEPT ONCE (same across trials) =====
        'bin_centers': preproc_data[0]['bin_centers'],
        'med_coords': preproc_data[0]['med_coords'],
        'stat_serializable': preproc_data[0].get('stat_serializable', None),
        'ops_serializable': preproc_data[0].get('ops_serializable', None),
        'processing_params': preproc_data[0].get('processing_params', None),
        
        # ===== METADATA =====
        'num_recordings': num_trials,
        'trials_per_recording': trials_per_recording,
        'total_trials': total_trials,
        'trial_info': trial_info,
        'session_directory': session_directory,
        'combination_timestamp': datetime.datetime.now().isoformat()
    }
    
    print(f"\nCombined data structure created with {len(combined_data)} fields")
    
    # Save combined data
    if save_output:
        print("\n" + "="*80)
        print("STEP 5: SAVING COMBINED SESSION DATA")
        print("="*80)
        
        session_name = os.path.basename(session_directory)
        output_filename = os.path.join(session_directory, f'{session_name}_combined_session.h5')
        
        print(f"\nSaving to: {output_filename}")
        
        try:
            files.write_h5(output_filename, combined_data)
            print("✓ Successfully saved combined session data!")
        except Exception as e:
            print(f"ERROR saving combined data: {e}")
            print("Attempting to save without problematic fields...")
            
            # Try saving without stat and ops
            minimal_data = {k: v for k, v in combined_data.items() 
                          if k not in ['stat_serializable', 'ops_serializable']}
            
            try:
                files.write_h5(output_filename, minimal_data)
                print("✓ Saved minimal combined data (without stat/ops)")
            except Exception as e2:
                print(f"ERROR: Could not save even minimal data: {e2}")
    
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"Session: {os.path.basename(session_directory)}")
    print(f"Recordings combined: {num_trials}")
    print(f"Total cells: {n_cells}")
    print(f"Total trials: {total_trials}")
    print(f"Spatial bins: {combined_spatial_activity.shape[1]}")
    print(f"\nReliable cells (reliable_cells in ALL recordings): {np.sum(session_reliable_cells)}")
    print(f"  ({100*np.sum(session_reliable_cells)/n_cells:.1f}% of all cells)")
    print(f"\nReliable cells (combined_reliable in ALL recordings): {np.sum(session_combined_reliable)}")
    print(f"  ({100*np.sum(session_combined_reliable)/n_cells:.1f}% of all cells)")
    print("="*80)
    
    return combined_data


if __name__ == "__main__":
    # Example usage
    session_directory = r"F:\2P\spmod\JSY052_ChrnoicImaging\251012_JSY_JSY052_SpatialModulation_Day4"
    
    combined_data = combine_session_trials(session_directory, save_output=True)
    
    # Optional: Print some statistics about the combined data
    print("\n" + "="*80)
    print("DETAILED STATISTICS")
    print("="*80)
    
    
    print(f"\nSession-level combined reliable cells statistics:")
    creliable_cells_mask = combined_data['session_reliable_cells']
    reliable_activity = combined_data['spatial_activity'][creliable_cells_mask, :, :]
    
    print(f"  Number of reliable cells: {np.sum(creliable_cells_mask)}")
    print(f"  Mean activity (reliable cells): {np.mean(reliable_activity):.4f}")
    print(f"  Std activity (reliable cells): {np.std(reliable_activity):.4f}")
    
    print(f"\nSession-level average correlation:")
    reliable_cc = combined_data['avg_cc_session'][creliable_cells_mask]
    print(f"  Mean CC (reliable cells): {np.mean(reliable_cc):.3f}")
    print(f"  Median CC (reliable cells): {np.median(reliable_cc):.3f}")
    
    print(f"\nSession-level Cohen's d:")
    reliable_cohen = combined_data['cohen_d_session'][creliable_cells_mask]
    print(f"  Mean Cohen's d (reliable cells): {np.mean(reliable_cohen):.3f}")
    print(f"  Median Cohen's d (reliable cells): {np.median(reliable_cohen):.3f}")
    
    print("\n" + "="*80)

    print(f"\nSession-level combined reliable cells statistics:")
    combined_reliable_mask = combined_data['session_combined_reliable']
    reliable_activity = combined_data['spatial_activity'][combined_reliable_mask, :, :]
    
    print(f"  Number of reliable cells: {np.sum(combined_reliable_mask)}")
    print(f"  Mean activity (reliable cells): {np.mean(reliable_activity):.4f}")
    print(f"  Std activity (reliable cells): {np.std(reliable_activity):.4f}")
    
    print(f"\nSession-level average correlation:")
    reliable_cc = combined_data['avg_cc_session'][combined_reliable_mask]
    print(f"  Mean CC (reliable cells): {np.mean(reliable_cc):.3f}")
    print(f"  Median CC (reliable cells): {np.median(reliable_cc):.3f}")
    
    print(f"\nSession-level Cohen's d:")
    reliable_cohen = combined_data['cohen_d_session'][combined_reliable_mask]
    print(f"  Mean Cohen's d (reliable cells): {np.mean(reliable_cohen):.3f}")
    print(f"  Median Cohen's d (reliable cells): {np.median(reliable_cohen):.3f}")


Found 2 trial recordings

Recording 1: TSeries-09072025-1257-001
  Loaded: 09072025_JSY038_preproc.h5
  Cells: 1017, Bins: 56, Trials: 130

Recording 2: TSeries-09072025-1257-002
  Loaded: 09072025_JSY038_preproc.h5
  Cells: 1017, Bins: 45, Trials: 130

✓ All recordings have 1017 cells (Suite2p concatenation successful)

STEP 1: CONCATENATING SPATIAL ACTIVITY DATA

Spatial activity shapes:
  Recording 1: (1017, 56, 130) (cells × bins × trials)
  Recording 2: (1017, 45, 130) (cells × bins × trials)

Combined spatial activity shape: (1017, 101, 130)
  (1017 cells × 101 bins × 130 trials)
  Bins per recording: [56, 45]

Combined spatial activity shape: (1017, 101, 130)
  (1017 cells × 101 trials × 130 bins)
  Trials per recording: [56, 45]

Combined spatial activity shape: (1017, 101, 130)
  (1017 cells × 101 bins × 130 total trials)
  Trials per recording: [130, 130]

STEP 2: AVERAGING PER-TRIAL STATISTICS

Averaged correlation coefficients across 2 recordings
  Mean CC range: [-0.116, 